# Chapter 20 — Change of Variables

Companion notebook to Chapter 20 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion. Jacobian determinants are computed by automatic
differentiation (`torch.func.jacrev`), never by hand.

Integrals are computed by the midpoint rule on fine grids. Seeded random
numbers are used for test points and, in the cells that say so, for
samples.

In [1]:
import math
import warnings

import torch
from torch.func import jacrev, vmap

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor


def midpoints(a, b, n):
    h = (b - a) / n
    return a + h * (torch.arange(n) + 0.5), h


def integral(f, a, b, n=2_000_000):
    x, h = midpoints(a, b, n)
    return (f(x) * h).sum()


def grid2(a, b, c, d, n):
    # midpoints of an n x n grid of [a, b] x [c, d], as an (n*n, 2) array, and the cell area
    x, hx = midpoints(a, b, n)
    y, hy = midpoints(c, d, n)
    X, Y = torch.meshgrid(x, y, indexing="ij")
    return torch.stack([X.flatten(), Y.flatten()], dim=1), hx * hy


def abs_det(g, pts):
    return vmap(lambda x: torch.linalg.det(jacrev(g)(x)).abs())(pts)


phi = lambda t: torch.exp(-t**2 / 2) / math.sqrt(2 * math.pi)

2.15.0+cu132


## 1. The theorem on a diffeomorphism of the unit square (Theorem 20.1, Corollary 20.5)

$g(u, v) = (u + 0.3\sin v,\; v + 0.3u^2)$ on $U = (0, 1)^2$. It is the
identity plus a map with Lipschitz constant $0.6 < 1$, hence injective
(Exercise 10.9), and $\det Dg = 1 - 0.18\,u\cos v > 0$.

The right-hand side $\int_U f(g(x))|\det Dg(x)|\,dx$ is computed on a grid
of $U$. The left-hand side $\int_V f(y)\,dy$ is computed independently on a
grid of a box containing $V = g(U)$, deciding whether a point $y$ is in $V$
by inverting $g$ with the fixed-point iteration $x\leftarrow y - F(x)$ of
Chapter 10.

In [2]:
def g(x):
    return torch.stack([x[..., 0] + 0.3 * torch.sin(x[..., 1]), x[..., 1] + 0.3 * x[..., 0]**2], dim=-1)


def g_inverse(y, iterations=80):
    x = y.clone()
    for _ in range(iterations):                    # x <- y - F(x), a contraction of ratio 0.6
        x = torch.stack([y[..., 0] - 0.3 * torch.sin(x[..., 1]), y[..., 1] - 0.3 * x[..., 0]**2], dim=-1)
    return x


f = lambda y: y[..., 0]**2 + torch.cos(3 * y[..., 1])

ptsU, cellU = grid2(0.0, 1.0, 0.0, 1.0, 700)
J = abs_det(g, ptsU)
assert torch.allclose(J, 1 - 0.18 * ptsU[:, 0] * torch.cos(ptsU[:, 1]))        # autograd = hand
rhs = (f(g(ptsU)) * J * cellU).sum()
area_rhs = (J * cellU).sum()

ptsB, cellB = grid2(-0.1, 1.4, -0.1, 1.4, 1500)                                # a box containing V
pre = g_inverse(ptsB)
assert (g(pre) - ptsB).abs().max() < 1e-12                                     # the inverse is exact
inV = ((pre > 0) & (pre < 1)).all(dim=1).double()
lhs = (f(ptsB) * inV * cellB).sum()
area_lhs = (inV * cellB).sum()

print(f"area of g(U):   grid of V = {area_lhs:.5f}   int_U |det Dg| = {area_rhs:.7f}"
      f"   exact = {1 - 0.09 * math.sin(1):.7f}")
print(f"int_V f dy = {lhs:.5f}      int_U f(g(x)) |det Dg(x)| dx = {rhs:.7f}")
assert abs(area_rhs - (1 - 0.09 * math.sin(1))) < 1e-6
assert abs(area_lhs - area_rhs) < 2e-3 and abs(lhs - rhs) < 2e-3

area of g(U):   grid of V = 0.92426   int_U |det Dg| = 0.9242676   exact = 0.9242676
int_V f dy = 0.30998      int_U f(g(x)) |det Dg(x)| dx = 0.3099760


The idea of the proof (Lemmas 20.2 and 20.3, Figure 20.1): the image of a
small square around $a$ has area close to $|\det Dg(a)|$ times the area of
the square. The area of the image is computed from its boundary by the
shoelace formula.

In [3]:
def image_area(center, half, m=4000):
    t = torch.linspace(-half, half, m + 1)[:-1]
    c0, c1 = center
    sides = [torch.stack([c0 + t, torch.full_like(t, c1 - half)], 1),
             torch.stack([torch.full_like(t, c0 + half), c1 + t], 1),
             torch.stack([c0 - t, torch.full_like(t, c1 + half)], 1),
             torch.stack([torch.full_like(t, c0 - half), c1 - t], 1)]
    b = g(torch.cat(sides))                                    # image of the boundary, counter-clockwise
    x, y = b[:, 0], b[:, 1]
    return 0.5 * (x * y.roll(-1) - y * x.roll(-1)).sum().abs().item()


a = T([0.7, 0.4])
det_a = torch.linalg.det(jacrev(g)(a)).abs().item()
previous = math.inf
for half in (0.2, 0.05, 0.0125, 0.003):
    ratio = image_area((0.7, 0.4), half) / (2 * half)**2
    print(f"half-side {half:6.4f}:  area of image / area of square = {ratio:.7f}")
    assert abs(ratio - det_a) < previous
    previous = abs(ratio - det_a)
print(f"|det Dg(a)|                                  = {det_a:.7f}")
assert previous < 1e-5

half-side 0.2000:  area of image / area of square = 0.8847185
half-side 0.0500:  area of image / area of square = 0.8839947
half-side 0.0125:  area of image / area of square = 0.8839493
half-side 0.0030:  area of image / area of square = 0.8839465
|det Dg(a)|                                  = 0.8839463


## 2. Polar coordinates (§20.3, Exercise 20.2)

The Jacobian determinant of $P(r, \theta) = (r\cos\theta, r\sin\theta)$ by
automatic differentiation, then $\int_{\mathbb R^2}e^{-|x|^2}dx = \pi$ and
$\int_{\mathbb R^2}(1 + |x|^2)^{-2}dx = \pi$, each computed on a Cartesian
grid and in polar coordinates.

In [4]:
P = lambda q: torch.stack([q[0] * torch.cos(q[1]), q[0] * torch.sin(q[1])])
pts, _ = grid2(0.1, 3.0, -3.0, 3.0, 30)
assert torch.allclose(abs_det(P, pts), pts[:, 0])
print("|det DP(r, theta)| = r at 900 points")

cart, cell = grid2(-8.0, 8.0, -8.0, 8.0, 2000)
gauss_cart = (torch.exp(-(cart**2).sum(1)) * cell).sum().item()
gauss_polar = 2 * math.pi * integral(lambda r: r * torch.exp(-r**2), 0.0, 10.0).item()
print(f"int exp(-|x|^2):  Cartesian grid = {gauss_cart:.8f}   polar = {gauss_polar:.8f}   pi = {math.pi:.8f}")
assert abs(gauss_cart - math.pi) < 1e-8 and abs(gauss_polar - math.pi) < 1e-8

rational_polar = 2 * math.pi * integral(lambda r: r / (1 + r**2)**2, 0.0, 4000.0, 8_000_000).item()
cart, cell = grid2(-60.0, 60.0, -60.0, 60.0, 3000)
rational_box = (cell / (1 + (cart**2).sum(1))**2).sum().item()        # the part inside a large box
print(f"int (1+|x|^2)^-2: polar = {rational_polar:.6f} (tail beyond r = 4000 is {math.pi / (1 + 4000**2):.1e})"
      f"   box [-60, 60]^2 = {rational_box:.5f}")
assert abs(rational_polar - math.pi) < 1e-6 and math.pi - 0.002 < rational_box < math.pi

# Exercise 20.2 (2) in R^3: int_{eps < |x| < 1} |x|^-a dx = 4 pi (1 - eps^(3-a)) / (3 - a)
for a_exp in (2.0, 2.9, 3.5):
    vals = [4 * math.pi * (1 - eps**(3 - a_exp)) / (3 - a_exp) for eps in (1e-2, 1e-4, 1e-8)]
    print(f"a = {a_exp}:  integrals over eps < |x| < 1 in R^3, eps = 1e-2, 1e-4, 1e-8:",
          [f"{v:.4g}" for v in vals])
    assert (vals[-1] < 1.01 * 4 * math.pi / (3 - a_exp)) if a_exp < 3 else (vals[-1] > 1e3)

|det DP(r, theta)| = r at 900 points
int exp(-|x|^2):  Cartesian grid = 3.14159265   polar = 3.14159265   pi = 3.14159265


int (1+|x|^2)^-2: polar = 3.141593 (tail beyond r = 4000 is 2.0e-07)   box [-60, 60]^2 = 3.14088
a = 2.0:  integrals over eps < |x| < 1 in R^3, eps = 1e-2, 1e-4, 1e-8: ['12.44', '12.57', '12.57']
a = 2.9:  integrals over eps < |x| < 1 in R^3, eps = 1e-2, 1e-4, 1e-8: ['46.38', '75.64', '105.7']
a = 3.5:  integrals over eps < |x| < 1 in R^3, eps = 1e-2, 1e-4, 1e-8: ['226.2', '2488', '2.513e+05']


## 3. How densities transform (Theorem 20.6, Figure 20.2, Exercise 20.4)

$X\sim N(m, s^2)$ and $Y = e^X$. The density of $Y$ is
$q(y) = p(\ln y)/y$. Probabilities of corresponding sets agree; the mode
does not follow: it is $e^{m - s^2}$, not $e^m$. The median does.

In [5]:
def lognormal(y, m, s):
    return torch.exp(-(torch.log(y) - m)**2 / (2 * s**2)) / (y * s * math.sqrt(2 * math.pi))


# the automatic version of the formula: q(g(x)) = p(x) / |g'(x)|
x = torch.linspace(-3, 3, 13)
dg = vmap(jacrev(torch.exp))(x)
assert torch.allclose(lognormal(torch.exp(x), 0.0, 1.0), phi(x) / dg.abs())

for m, s in [(0.0, 1.0), (0.4, 0.6)]:
    mass = integral(lambda u: lognormal(torch.exp(u), m, s) * torch.exp(u), -30.0, 30.0).item()   # y = e^u
    a_, b_ = m - 0.5, m + 0.5
    prob_x = integral(lambda t: torch.exp(-(t - m)**2 / (2 * s**2)) / (s * math.sqrt(2 * math.pi)), a_, b_).item()
    prob_y = integral(lambda y: lognormal(y, m, s), math.exp(a_), math.exp(b_)).item()
    ys = torch.linspace(1e-3, 6.0, 2_000_001)
    mode = ys[torch.argmax(lognormal(ys, m, s))].item()
    below_median = integral(lambda y: lognormal(y, m, s), 1e-12, math.exp(m), 8_000_000).item()
    print(f"m = {m}, s = {s}:  mass = {mass:.8f}   P(X in [m-1/2, m+1/2]) = {prob_x:.7f} = P(Y in image) = {prob_y:.7f}")
    print(f"      mode of Y = {mode:.4f}   e^(m - s^2) = {math.exp(m - s**2):.4f}   image of the mode of X = {math.exp(m):.4f}"
          f"   P(Y <= e^m) = {below_median:.5f}")
    assert abs(mass - 1) < 1e-8 and abs(prob_x - prob_y) < 1e-7
    assert abs(mode - math.exp(m - s**2)) < 1e-4 and abs(below_median - 0.5) < 1e-4

# the heights quoted in Figure 20.2
assert abs(lognormal(T(math.exp(-1.0)), 0.0, 1.0) - 0.658) < 1e-3 and abs(lognormal(T(1.0), 0.0, 1.0) - 0.399) < 1e-3

m = 0.0, s = 1.0:  mass = 1.00000000   P(X in [m-1/2, m+1/2]) = 0.3829249 = P(Y in image) = 0.3829249
      mode of Y = 0.3679   e^(m - s^2) = 0.3679   image of the mode of X = 1.0000   P(Y <= e^m) = 0.50000


m = 0.4, s = 0.6:  mass = 1.00000000   P(X in [m-1/2, m+1/2]) = 0.5953432 = P(Y in image) = 0.5953432
      mode of Y = 1.0408   e^(m - s^2) = 1.0408   image of the mode of X = 1.4918   P(Y <= e^m) = 0.50000


**Exercise 20.3.** $-\ln U$ is exponential and $U^2$ has density
$1/(2\sqrt y)$; the Box–Muller map sends the uniform density on the square
to the standard Gaussian density on the plane: $1/|\det Dg(u)|$, by
automatic differentiation, equals $\varphi(z_1)\varphi(z_2)$ at $z = g(u)$.
A seeded sample confirms the first two.

In [6]:
def box_muller(u):
    r = torch.sqrt(-2 * torch.log(u[0]))
    return torch.stack([r * torch.cos(2 * math.pi * u[1]), r * torch.sin(2 * math.pi * u[1])])


u = 0.02 + 0.96 * torch.rand(2000, 2)
z = vmap(box_muller)(u)
density_z = 1.0 / abs_det(box_muller, u)                      # p(u) = 1 on the square
assert torch.allclose(density_z, phi(z[:, 0]) * phi(z[:, 1]), rtol=1e-9)
print("Box-Muller: 1 / |det Dg(u)| = phi(z1) phi(z2) at 2000 points")

U = torch.rand(2_000_000)
for name, sample, cdf in [("-ln U", -torch.log(U), lambda t: 1 - math.exp(-t)),
                          ("U^2", U**2, lambda t: math.sqrt(t))]:
    for t in (0.1, 0.5, 0.9):
        emp = (sample <= t).double().mean().item()
        assert abs(emp - cdf(t)) < 2e-3
    print(f"{name:6s} empirical distribution function matches the one of the stated density")

Box-Muller: 1 / |det Dg(u)| = phi(z1) phi(z2) at 2000 points
-ln U  empirical distribution function matches the one of the stated density
U^2    empirical distribution function matches the one of the stated density


**Example 20.7 and Exercise 20.5.** The density of $x = m + Lz$ from the
change of variables formula, with the Cholesky factor, against PyTorch's
multivariate normal.

In [7]:
n = 5
A = torch.randn(n, n)
Sigma = A @ A.T + 0.5 * torch.eye(n)
m = torch.randn(n)
L = torch.linalg.cholesky(Sigma)
xs = m + torch.randn(8, n) @ L.T

w = torch.linalg.solve_triangular(L, (xs - m).T, upper=False).T          # L^{-1}(x - m)
log_q = -0.5 * (w**2).sum(1) - torch.log(torch.diagonal(L)).sum() - 0.5 * n * math.log(2 * math.pi)
reference = torch.distributions.MultivariateNormal(m, covariance_matrix=Sigma).log_prob(xs)
print("change of variables:", [f"{v:.5f}" for v in log_q.tolist()[:4]])
print("torch.distributions:", [f"{v:.5f}" for v in reference.tolist()[:4]])
assert torch.allclose(log_q, reference)
assert abs(2 * torch.log(torch.diagonal(L)).sum() - torch.logdet(Sigma)) < 1e-10

change of variables: ['-11.50897', '-10.53754', '-10.13832', '-9.16779']
torch.distributions: ['-11.50897', '-10.53754', '-10.13832', '-9.16779']


## 4. The volume of a submanifold (Definition 20.8, Theorem 20.9, Example 20.10, Exercises 20.1 and 20.7)

$\sqrt{\det G}$ with $G = D\psi^\top D\psi$ computed by automatic
differentiation. The area of the unit sphere in two parametrizations, the
area of a paraboloid, the length of a curve, and the area of an ellipse.

In [8]:
def sqrt_gram(psi, pts):
    def one(u):
        D = jacrev(psi)(u)
        return torch.sqrt(torch.linalg.det(D.T @ D))
    return vmap(one)(pts)


lonlat = lambda u: torch.stack([torch.cos(u[1]) * torch.cos(u[0]), torch.cos(u[1]) * torch.sin(u[0]), torch.sin(u[1])])
pts, cell = grid2(-math.pi, math.pi, -math.pi / 2, math.pi / 2, 500)
sg = sqrt_gram(lonlat, pts)
assert torch.allclose(sg, torch.cos(pts[:, 1]))
area1 = (sg * cell).sum().item()


def stereo(u):
    s = (u**2).sum()
    return torch.cat([2 * u, (s - 1).reshape(1)]) / (s + 1)


pts, cell = grid2(-60.0, 60.0, -60.0, 60.0, 1500)
sg = sqrt_gram(stereo, pts)
assert torch.allclose(sg, 4 / (1 + (pts**2).sum(1))**2)                  # G = 4 (1 + |u|^2)^-2 I
area2_box = (sg * cell).sum().item()
area2 = 4 * 2 * math.pi * integral(lambda r: r / (1 + r**2)**2, 0.0, 4000.0, 8_000_000).item()
print(f"area of the unit sphere: longitude-latitude = {area1:.6f}   stereographic = {area2:.6f}"
      f"   (box |u| < 60: {area2_box:.4f})   4 pi = {4 * math.pi:.6f}")
assert abs(area1 - 4 * math.pi) < 1e-4 and abs(area2 - 4 * math.pi) < 1e-5
assert 4 * math.pi - 0.01 < area2_box < 4 * math.pi

# The same region of the sphere measured in both charts: the cap z > 1/2.
pts, cell = grid2(-math.pi, math.pi, -math.pi / 2, math.pi / 2, 1200)
cap1 = (torch.cos(pts[:, 1]) * (torch.sin(pts[:, 1]) > 0.5) * cell).sum().item()
pts, cell = grid2(-60.0, 60.0, -60.0, 60.0, 3000)
s2 = (pts**2).sum(1)
cap2 = (4 / (1 + s2)**2 * (((s2 - 1) / (s2 + 1)) > 0.5) * cell).sum().item()
tail = 4 * math.pi / (1 + 60.0**2)                                       # part of the plane outside the box, at most
print(f"area of the cap z > 1/2: {cap1:.4f} and {cap2:.4f} (+ at most {tail:.4f})   exact = pi = {math.pi:.4f}")
assert abs(cap1 - math.pi) < 5e-3 and abs(cap2 - math.pi) < 0.02

area of the unit sphere: longitude-latitude = 12.566391   stereographic = 12.566370   (box |u| < 60: 12.5635)   4 pi = 12.566371


area of the cap z > 1/2: 3.1416 and 3.1413 (+ at most 0.0035)   exact = pi = 3.1416


In [9]:
# Paraboloid z = (x^2 + y^2)/2 over the unit disc, as a graph in polar parameters.
parab = lambda q: torch.stack([q[0] * torch.cos(q[1]), q[0] * torch.sin(q[1]), q[0]**2 / 2])
pts, cell = grid2(0.0, 1.0, -math.pi, math.pi, 400)
area = (sqrt_gram(parab, pts) * cell).sum().item()
exact = 2 * math.pi / 3 * (2 * math.sqrt(2) - 1)
print(f"area of the paraboloid = {area:.6f}   (2 pi / 3)(2 sqrt 2 - 1) = {exact:.6f}")
assert abs(area - exact) < 1e-5

# det(I + grad grad^T) = 1 + |grad|^2 for a graph (Example 20.10 (3))
fgraph = lambda u: torch.cat([u, (torch.sin(u[0]) * u[1]**2 + u[2]).reshape(1)])
u = torch.randn(50, 3)
grad = torch.stack([torch.cos(u[:, 0]) * u[:, 1]**2, 2 * torch.sin(u[:, 0]) * u[:, 1], torch.ones(50)], 1)
assert torch.allclose(sqrt_gram(fgraph, u)**2, 1 + (grad**2).sum(1))

# A curve: the length of one turn of a helix is 2 pi sqrt(1 + c^2).
helix = lambda t: torch.stack([torch.cos(t[0]), torch.sin(t[0]), 0.5 * t[0]])
t, h = midpoints(0.0, 2 * math.pi, 2000)
length = (sqrt_gram(helix, t[:, None]) * h).sum().item()
print(f"length of a helix turn = {length:.8f}   2 pi sqrt(1.25) = {2 * math.pi * math.sqrt(1.25):.8f}")
assert abs(length - 2 * math.pi * math.sqrt(1.25)) < 1e-9

# Exercise 20.1: ellipse and ellipsoid.
a_, b_ = 2.0, 0.7
pts, cell = grid2(-a_, a_, -b_, b_, 2000)
ellipse = ((pts[:, 0] / a_)**2 + (pts[:, 1] / b_)**2 <= 1).double().sum().item() * cell
print(f"area of the ellipse = {ellipse:.4f}   pi a b = {math.pi * a_ * b_:.4f}")
assert abs(ellipse - math.pi * a_ * b_) < 5e-3

area of the paraboloid = 3.829447   (2 pi / 3)(2 sqrt 2 - 1) = 3.829449
length of a helix turn = 7.02481473   2 pi sqrt(1.25) = 7.02481473
area of the ellipse = 4.3983   pi a b = 4.3982


The ellipsoid $\{x^\top\Sigma^{-1}x\leq1\}$ in $\mathbb R^3$ has volume
$\sqrt{\det\Sigma}\,\omega_3$; checked with a seeded uniform sample of a
box.

In [10]:
A3 = torch.randn(3, 3)
S3 = A3 @ A3.T + 0.3 * torch.eye(3)
S3inv = torch.linalg.inv(S3)
half = torch.sqrt(torch.diagonal(S3))                         # the ellipsoid lies in the box |x_i| <= sqrt(S_ii)
M = 4_000_000
x = (2 * torch.rand(M, 3) - 1) * half
frac = (((x @ S3inv) * x).sum(1) <= 1).double().mean().item()
vol = frac * (2 * half).prod().item()
exact = math.sqrt(torch.linalg.det(S3).item()) * 4 * math.pi / 3
se = (2 * half).prod().item() * math.sqrt(frac * (1 - frac) / M)
print(f"volume of the ellipsoid = {vol:.4f} +- {se:.4f}   sqrt(det Sigma) * 4 pi / 3 = {exact:.4f}")
assert abs(vol - exact) < 5 * se

volume of the ellipsoid = 10.5717 +- 0.0092   sqrt(det Sigma) * 4 pi / 3 = 10.5667


**Proposition 20.12.** A Gaussian latent variable pushed onto the parabola
$g(z) = (z, z^2)$: the image has no density in the plane, but with respect
to the length of the curve its density is $p(z)/\|g'(z)\|$. The probability
of an arc, computed from that density and the length element, agrees with a
seeded sample.

In [11]:
curve = lambda t: torch.stack([t[0], t[0]**2])
z, h = midpoints(0.5, 1.5, 200_000)                            # the arc between z = 0.5 and z = 1.5
speed = sqrt_gram(curve, z[:, None])                           # ||g'(z)|| = sqrt(1 + 4 z^2)
assert torch.allclose(speed, torch.sqrt(1 + 4 * z**2))
density_on_curve = phi(z) / speed
prob = (density_on_curve * speed * h).sum().item()            # integral for the length measure
length = (speed * h).sum().item()
sample = torch.randn(2_000_000)
emp = ((sample > 0.5) & (sample < 1.5)).double().mean().item()
print(f"length of the arc = {length:.5f}   probability of the arc = {prob:.5f}   sample = {emp:.5f}")
assert abs(prob - emp) < 2e-3

length of the arc = 2.25242   probability of the arc = 0.24173   sample = 0.24204


## 5. Spheres, balls and high dimension (Theorem 20.13, Corollary 20.14, Figure 20.4, Exercise 20.8)

The formulas $\sigma(S^{n-1}) = 2\pi^{n/2}/\Gamma(n/2)$ and
$\omega_n = \pi^{n/2}/\Gamma(n/2 + 1)$; the volume of the ball against a
seeded uniform sample of the cube; and the radial formula of Theorem 20.13
on a function that is not radial, in $\mathbb R^3$.

In [12]:
area = lambda n: 2 * math.pi**(n / 2) / math.gamma(n / 2)
ball = lambda n: math.pi**(n / 2) / math.gamma(n / 2 + 1)
assert abs(area(2) - 2 * math.pi) < 1e-12 and abs(area(3) - 4 * math.pi) < 1e-12 and abs(area(4) - 2 * math.pi**2) < 1e-12
assert abs(ball(2) - math.pi) < 1e-12 and abs(ball(3) - 4 * math.pi / 3) < 1e-12 and abs(ball(4) - math.pi**2 / 2) < 1e-12
print("n :", list(range(1, 13)))
print("volume of the unit ball:", [f"{ball(n):.3f}" for n in range(1, 13)])
assert max(range(1, 40), key=ball) == 5 and ball(40) < 1e-8
print("fraction of the cube [-1, 1]^n filled by the ball:", [f"{ball(n) / 2**n:.1e}" for n in (2, 5, 10, 20)])

M = 4_000_000
for n in (2, 3, 4, 5, 6):
    frac = ((2 * torch.rand(M, n) - 1).pow(2).sum(1) <= 1).double().mean().item()
    se = 2**n * math.sqrt(frac * (1 - frac) / M)
    print(f"n = {n}:  sampled volume = {frac * 2**n:.4f} +- {se:.4f}   formula = {ball(n):.4f}")
    assert abs(frac * 2**n - ball(n)) < 5 * se

# Theorem 20.13 in R^3 for f(x) = exp(-|x|^2) (1 + x_3)^2:
# Cartesian grid against  int_0^inf ( int_sphere f(r w) dsigma(w) ) r^2 dr  in the longitude-latitude chart.
f3 = lambda x: torch.exp(-(x**2).sum(-1)) * (1 + x[..., 2])**2
g1, h1 = midpoints(-6.0, 6.0, 240)
X, Y, Z = torch.meshgrid(g1, g1, g1, indexing="ij")
cartesian = (f3(torch.stack([X, Y, Z], -1)) * h1**3).sum().item()

ang, cell = grid2(-math.pi, math.pi, -math.pi / 2, math.pi / 2, 200)
omega = vmap(lonlat)(ang)                                       # points of the sphere
weight = torch.cos(ang[:, 1]) * cell                            # sqrt(det G) du
r, hr = midpoints(0.0, 8.0, 2000)
inner = (f3(r[:, None, None] * omega[None]) * weight[None]).sum(1)      # int over the sphere, for each r
polar = (inner * r**2 * hr).sum().item()
exact = math.pi**1.5 * 1.5                                      # pi^(3/2) (1 + E[x_3^2]) with variance 1/2
print(f"Cartesian = {cartesian:.6f}   polar = {polar:.6f}   exact = {exact:.6f}")
assert abs(cartesian - exact) < 1e-6 and abs(polar - exact) < 1e-3

n : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
volume of the unit ball: ['2.000', '3.142', '4.189', '4.935', '5.264', '5.168', '4.725', '4.059', '3.299', '2.550', '1.884', '1.335']
fraction of the cube [-1, 1]^n filled by the ball: ['7.9e-01', '1.6e-01', '2.5e-03', '2.5e-08']
n = 2:  sampled volume = 3.1416 +- 0.0008   formula = 3.1416


n = 3:  sampled volume = 4.1863 +- 0.0020   formula = 4.1888


n = 4:  sampled volume = 4.9339 +- 0.0037   formula = 4.9348


n = 5:  sampled volume = 5.2588 +- 0.0059   formula = 5.2638


n = 6:  sampled volume = 5.1695 +- 0.0087   formula = 5.1677


Cartesian = 8.352492   polar = 8.352635   exact = 8.352492


The first machine-learning box of §20.5. The shell $0.95\leq r\leq1$ of a
ball in dimension $100$ holds more than $99\%$ of its volume. The radius of
a standard Gaussian point has density proportional to $r^{n-1}e^{-r^2/2}$,
with mode $\sqrt{n-1}$ and a width that does not grow; the constants of
Figure 20.4 are checked; and seeded samples show codes of norm about
$\sqrt n$ whose midpoints have norm about $\sqrt{n/2}$.

In [13]:
print(f"(1 - 0.05)^100 = {0.95**100:.5f}")
assert 0.95**100 < 0.01


def log_const(n):
    # ln( sigma(S^{n-1}) (2 pi)^{-n/2} ), with lgamma to avoid overflow in high dimension
    return math.log(2) + n / 2 * math.log(math.pi) - math.lgamma(n / 2) - n / 2 * math.log(2 * math.pi)


def radial_density(r, n):
    return torch.exp(log_const(n) + (n - 1) * torch.log(r) - r**2 / 2)


r, h = midpoints(1e-9, 40.0, 4_000_000)
for n, drawn in [(2, 0.0), (10, 5.95064), (50, 71.42026), (500, None)]:
    d = radial_density(r, n)
    mass = (d * h).sum().item()
    mode = r[torch.argmax(d)].item()
    mean = (r * d * h).sum().item()
    sd = math.sqrt((r**2 * d * h).sum().item() - mean**2)
    print(f"n = {n:3d}:  mass = {mass:.8f}   mode = {mode:.4f} (sqrt(n-1) = {math.sqrt(n - 1):.4f})"
          f"   mean radius = {mean:.4f}   standard deviation = {sd:.4f}")
    assert abs(mass - 1) < 1e-8 and abs(mode - math.sqrt(n - 1)) < 1e-4 and sd < 0.72
    assert abs((r**2 * d * h).sum().item() - n) < 1e-6             # E|x|^2 = n
    if drawn is not None:                                          # the constants used to draw Figure 20.4
        assert abs(-log_const(n) - drawn) < 1e-4

n = 400
z1, z2 = torch.randn(20_000, n), torch.randn(20_000, n)
print(f"n = {n}: mean norm of a code = {z1.norm(dim=1).mean():.3f} (sqrt n = {math.sqrt(n):.3f})   "
      f"mean norm of a midpoint = {((z1 + z2) / 2).norm(dim=1).mean():.3f} (sqrt(n/2) = {math.sqrt(n / 2):.3f})")
print(f"   smallest norm among 20000 codes = {z1.norm(dim=1).min():.3f}; the density is largest at norm 0")
assert abs(z1.norm(dim=1).mean() - math.sqrt(n)) < 0.1
assert abs(((z1 + z2) / 2).norm(dim=1).mean() - math.sqrt(n / 2)) < 0.1 and z1.norm(dim=1).min() > 17

(1 - 0.05)^100 = 0.00592
n =   2:  mass = 1.00000000   mode = 1.0000 (sqrt(n-1) = 1.0000)   mean radius = 1.2533   standard deviation = 0.6551


n =  10:  mass = 1.00000000   mode = 3.0000 (sqrt(n-1) = 3.0000)   mean radius = 3.0843   standard deviation = 0.6978


n =  50:  mass = 1.00000000   mode = 7.0000 (sqrt(n-1) = 7.0000)   mean radius = 7.0358   standard deviation = 0.7053
n = 500:  mass = 1.00000000   mode = 22.3383 (sqrt(n-1) = 22.3383)   mean radius = 22.3495   standard deviation = 0.7069


n = 400: mean norm of a code = 19.976 (sqrt n = 20.000)   mean norm of a midpoint = 14.130 (sqrt(n/2) = 14.142)
   smallest norm among 20000 codes = 17.326; the density is largest at norm 0


## 6. Normalizing flows (first box of §20.6, Exercise 20.6)

A coupling layer $y_1 = x_1$, $y_2 = x_2\odot e^{s(x_1)} + t(x_1)$ with
small random networks $s$ and $t$ on $\mathbb R^3\times\mathbb R^3$. The
log-determinant by automatic differentiation equals $\sum_i s_i(x_1)$; the
inverse is explicit; and for two layers the log-determinants add.

In [14]:
d = 3


def make_net():
    W1, b1, W2 = torch.randn(8, d), torch.randn(8), 0.5 * torch.randn(d, 8)
    return lambda v: W2 @ torch.tanh(W1 @ v + b1)


s1, t1, s2, t2 = make_net(), make_net(), make_net(), make_net()


def layer_a(x):                      # transforms the second half
    return torch.cat([x[:d], x[d:] * torch.exp(s1(x[:d])) + t1(x[:d])])


def layer_a_inv(y):
    return torch.cat([y[:d], (y[d:] - t1(y[:d])) * torch.exp(-s1(y[:d]))])


def layer_b(x):                      # transforms the first half
    return torch.cat([x[:d] * torch.exp(s2(x[d:])) + t2(x[d:]), x[d:]])


flow = lambda x: layer_b(layer_a(x))
for _ in range(5):
    x = torch.randn(2 * d)
    Ja = jacrev(layer_a)(x)
    assert torch.allclose(Ja[:d, :d], torch.eye(d)) and torch.allclose(Ja[:d, d:], torch.zeros(d, d))
    logdet_a = torch.linalg.slogdet(Ja).logabsdet
    assert abs(logdet_a - s1(x[:d]).sum()) < 1e-10
    assert torch.allclose(layer_a_inv(layer_a(x)), x)
    ya = layer_a(x)
    logdet_b = torch.linalg.slogdet(jacrev(layer_b)(ya)).logabsdet
    assert abs(logdet_b - s2(ya[d:]).sum()) < 1e-10
    total = torch.linalg.slogdet(jacrev(flow)(x)).logabsdet
    assert abs(total - (logdet_a + logdet_b)) < 1e-10
print(f"last point: log|det| of layer a = {logdet_a:+.5f}, of layer b = {logdet_b:+.5f}, of the composition = {total:+.5f}")

last point: log|det| of layer a = +1.87111, of layer b = -4.16179, of the composition = -2.29068


The two-dimensional layer of Exercise 20.6 (3): $y_1 = x_1$,
$y_2 = x_2e^{x_1} + x_1^2$ applied to a standard Gaussian. The density
$q(y) = \varphi(y_1)\,\varphi((y_2 - y_1^2)e^{-y_1})\,e^{-y_1}$ integrates
to $1$, and gives the probability of a box that a seeded sample confirms.

In [15]:
q = lambda y: phi(y[:, 0]) * phi((y[:, 1] - y[:, 0]**2) * torch.exp(-y[:, 0])) * torch.exp(-y[:, 0])

# the formula from automatic differentiation: q(g(x)) = p(x) / |det Dg(x)|
lay = lambda x: torch.stack([x[0], x[1] * torch.exp(x[0]) + x[0]**2])
x = torch.randn(500, 2)
assert torch.allclose(q(vmap(lay)(x)), phi(x[:, 0]) * phi(x[:, 1]) / abs_det(lay, x))

# integrate q over the plane in the variables (y1, w) with y2 = y1^2 + w e^{y1}, to resolve the thin ridge
yw, cell = grid2(-9.0, 9.0, -9.0, 9.0, 1500)
y1, wv = yw[:, 0], yw[:, 1]
y2 = y1**2 + wv * torch.exp(y1)
mass = (q(torch.stack([y1, y2], 1)) * torch.exp(y1) * cell).sum().item()
print(f"integral of q over the plane = {mass:.8f}")
assert abs(mass - 1) < 1e-8

box, cell = grid2(-0.5, 1.0, 0.0, 2.0, 1500)
prob = (q(box) * cell).sum().item()
xs = torch.randn(4_000_000, 2)
ys = torch.stack([xs[:, 0], xs[:, 1] * torch.exp(xs[:, 0]) + xs[:, 0]**2], 1)
emp = ((ys[:, 0] > -0.5) & (ys[:, 0] < 1.0) & (ys[:, 1] > 0.0) & (ys[:, 1] < 2.0)).double().mean().item()
print(f"P(y in [-0.5, 1] x [0, 2]):  from the density = {prob:.5f}   from samples = {emp:.5f}")
assert abs(prob - emp) < 1.5e-3

integral of q over the plane = 1.00000000


P(y in [-0.5, 1] x [0, 2]):  from the density = 0.24324   from samples = 0.24315


The last item of the box: the differential entropy of the output is that of
the input plus $\mathbb E[\ln|\det Dg(z)|]$. For this layer
$\ln|\det Dg(x)| = x_1$, of mean $0$, so the output has the entropy
$\ln(2\pi e)$ of the standard Gaussian of the plane.

In [16]:
qv = q(torch.stack([y1, y2], 1))                         # on the (y1, w) grid of the previous cell
_, cell_yw = grid2(-9.0, 9.0, -9.0, 9.0, 1500)
entropy = -(qv * torch.log(qv.clamp(min=1e-300)) * torch.exp(y1) * cell_yw).sum().item()
print(f"entropy of the output = {entropy:.6f}   ln(2 pi e) = {math.log(2 * math.pi * math.e):.6f}")
assert abs(entropy - math.log(2 * math.pi * math.e)) < 1e-6

entropy of the output = 2.837877   ln(2 pi e) = 2.837877


## 7. The volume of the Fisher metric (second box of §20.6, Remark 20.11, Exercise 20.9)

For a coin, $F(\theta) = 1/(\theta(1-\theta))$, obtained here from the
definition as the variance of the score. The total mass of
$\sqrt F\,d\theta$ is $\pi$, the length of the quarter circle of radius $2$.
In the parameter $\eta = \ln\frac\theta{1-\theta}$ the Fisher information
is $\theta(1-\theta)$, and $\sqrt{F_\eta}\,d\eta$ gives the same mass to
corresponding intervals. A uniform prior in $\theta$ is not uniform in
$\eta$.

In [17]:
def fisher(log_prob, param):
    # variance of the score for a variable with outcomes 1 and 0
    out = torch.zeros(())
    for outcome in (1.0, 0.0):
        lp = lambda t: log_prob(t, outcome)
        score = jacrev(lp)(param)
        out = out + torch.exp(lp(param)) * score**2
    return out


lp_theta = lambda th, o: o * torch.log(th) + (1 - o) * torch.log(1 - th)
lp_eta = lambda et, o: o * torch.nn.functional.logsigmoid(et) + (1 - o) * torch.nn.functional.logsigmoid(-et)

for th in (0.1, 0.5, 0.83):
    eta = math.log(th / (1 - th))
    F_theta, F_eta = fisher(lp_theta, T(th)).item(), fisher(lp_eta, T(eta)).item()
    assert abs(F_theta - 1 / (th * (1 - th))) < 1e-10 and abs(F_eta - th * (1 - th)) < 1e-10
    assert abs(F_eta - F_theta * (th * (1 - th))**2) < 1e-10          # F_eta = F (dtheta/deta)^2
print("F(theta) = 1/(theta(1-theta)),  F_eta = theta(1-theta) = F (dtheta/deta)^2")

# total mass: substitute theta = sin^2 t to remove the singularities at 0 and 1
mass = integral(lambda t: 2 * torch.ones_like(t), 0.0, math.pi / 2).item()
direct = integral(lambda th: 1 / torch.sqrt(th * (1 - th)), 0.0, 1.0, 8_000_000).item()
sig = torch.sigmoid
mass_eta = integral(lambda e: torch.sqrt(sig(e) * sig(-e)), -80.0, 80.0, 4_000_000).item()
print(f"total mass: theta = sin^2 t gives {mass:.8f};  midpoint rule in theta {direct:.4f};  in eta {mass_eta:.8f};  pi = {math.pi:.8f}")
assert abs(mass - math.pi) < 1e-10 and abs(mass_eta - math.pi) < 1e-8 and abs(direct - math.pi) < 5e-3

# the same measure in both parametrizations, on corresponding intervals
for a_, b_ in [(0.1, 0.3), (0.5, 0.9), (0.02, 0.98)]:
    in_theta = integral(lambda th: 1 / torch.sqrt(th * (1 - th)), a_, b_).item()
    ea, eb = math.log(a_ / (1 - a_)), math.log(b_ / (1 - b_))
    in_eta = integral(lambda e: torch.sqrt(sig(e) * sig(-e)), ea, eb).item()
    arc = 2 * (math.asin(math.sqrt(b_)) - math.asin(math.sqrt(a_)))     # length on the circle of radius 2
    print(f"theta in [{a_}, {b_}]:  int sqrt(F) dtheta = {in_theta:.7f}   int sqrt(F_eta) deta = {in_eta:.7f}   arc length = {arc:.7f}")
    assert abs(in_theta - in_eta) < 1e-7 and abs(in_theta - arc) < 1e-7

# a uniform prior in theta has the logistic density in eta
e = torch.linspace(-4, 4, 9)
dtheta_deta = vmap(jacrev(torch.sigmoid))(e)
assert torch.allclose(dtheta_deta, torch.exp(e) / (1 + torch.exp(e))**2)
print("density in eta of the prior uniform in theta:", [f"{v:.3f}" for v in dtheta_deta.tolist()])

F(theta) = 1/(theta(1-theta)),  F_eta = theta(1-theta) = F (dtheta/deta)^2


total mass: theta = sin^2 t gives 3.14159265;  midpoint rule in theta 3.1412;  in eta 3.14159265;  pi = 3.14159265
theta in [0.1, 0.3]:  int sqrt(F) dtheta = 0.5157784   int sqrt(F_eta) deta = 0.5157784   arc length = 0.5157784
theta in [0.5, 0.9]:  int sqrt(F) dtheta = 0.9272952   int sqrt(F_eta) deta = 0.9272952   arc length = 0.9272952
theta in [0.02, 0.98]:  int sqrt(F) dtheta = 2.5740044   int sqrt(F_eta) deta = 2.5740044   arc length = 2.5740044
density in eta of the prior uniform in theta: ['0.018', '0.045', '0.105', '0.197', '0.250', '0.197', '0.105', '0.045', '0.018']


## Summary

* $\int_V f = \int_U (f\circ g)|\det Dg|$ on a diffeomorphism of the square,
  with both sides computed independently; the image of a small square has
  area $|\det Dg(a)|$ times its own, in the limit.
* $\int e^{-|x|^2} = \pi$ and $\int(1 + |x|^2)^{-2} = \pi$ in the plane.
* The lognormal density has mass $1$, mode $e^{m - s^2}$ and median $e^m$;
  Box–Muller turns the uniform density into the Gaussian one; the Gaussian
  log-density is computed from a Cholesky factor.
* Areas from $\sqrt{\det(D\psi^\top D\psi)}$: the sphere ($4\pi$ in two
  charts), a cap ($\pi$ in two charts), a paraboloid, a helix.
* $\omega_n = \pi^{n/2}/\Gamma(n/2 + 1)$ is largest for $n = 5$; the radial
  density of a Gaussian peaks at $\sqrt{n - 1}$ with a width below $0.72$.
* Coupling layers: $\ln|\det| = \sum_i s_i(x_1)$, log-determinants add, and
  the output density integrates to $1$.
* $\sqrt F\,d\theta$ for a coin has total mass $\pi$ and is the same measure
  in $\theta$ and in the logit.